# HearSight Two-Stage Training

Standalone Kaggle notebook. Attach the two-stage dataset produced by `kaggle_dataset_builder.ipynb`, then run this notebook. It trains YOLO26n detector on GPU 0 and YOLO26n classifier on GPU 1 in parallel. No helper scripts are required.

In [1]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("RoboFlow_Key")
    secret_value_1 = user_secrets.get_secret("HF_TOKEN")
    os.environ["ROBOFLOW_API_KEY"] = secret_value_0
    os.environ["HF_TOKEN"] = secret_value_1
    print("Kaggle secrets loaded: RoboFlow_Key, HF_TOKEN")
except Exception as e:
    print("Kaggle secrets not available in this environment:", e)


Kaggle secrets loaded: RoboFlow_Key, HF_TOKEN


In [2]:
import importlib.util, subprocess, sys
packages = []
for module, package in [("ultralytics", "ultralytics"), ("cv2", "opencv-python-headless")]:
    if importlib.util.find_spec(module) is None:
        packages.append(package)
if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.5 MB/s eta 0:00:00


In [3]:
from pathlib import Path
import shutil, subprocess, time
import yaml
import urllib.request

def download(url, dst):
    dst = Path(dst)
    if not dst.exists():
        print("Downloading", dst.name)
        urllib.request.urlretrieve(url, dst)
    print(dst, dst.stat().st_size)

# Keep nano models for Raspberry Pi deployment. Quality comes from data, resolution, verifier, and thresholds.
download("https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt", "yolo26n.pt")
download("https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n-cls.pt", "yolo26n-cls.pt")


yolo26n.pt 5544453
yolo26n-cls.pt 5786434


In [4]:
from pathlib import Path
import json
import re
import shutil
import subprocess
import time
import yaml


IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
DISABLED_CLASSIFIER_CLASSES = [
    "Exit",
    "Fire extinguisher",
    "Traffic light red",
    "Traffic light yellow",
    "Traffic light green",
]
DISABLED_DETECTOR_CLASSES = {"traffic_light"}


def sanitize(name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", name).strip("_")


def load_yaml_names(data):
    names = data.get("names", [])
    if isinstance(names, dict):
        return [names[k] for k in sorted(names, key=lambda x: int(x))]
    return list(names)


def move_disabled_classifier_classes_to_not_target(dataset):
    dataset = Path(dataset)
    moved = {split: {name: 0 for name in DISABLED_CLASSIFIER_CLASSES} for split in ("train", "val", "test")}
    for split in ("train", "val", "test"):
        split_dir = dataset / "classifier_dataset" / split
        if not split_dir.exists():
            continue
        not_target_dir = split_dir / "not_target"
        not_target_dir.mkdir(parents=True, exist_ok=True)
        for class_name in DISABLED_CLASSIFIER_CLASSES:
            class_dir = split_dir / sanitize(class_name)
            if not class_dir.exists():
                continue
            for image_path in sorted(p for p in class_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS):
                dst = not_target_dir / f"disabled_{sanitize(class_name)}_{image_path.name}"
                i = 1
                while dst.exists():
                    dst = not_target_dir / f"disabled_{sanitize(class_name)}_{image_path.stem}_{i}{image_path.suffix.lower()}"
                    i += 1
                shutil.move(str(image_path), dst)
                moved[split][class_name] += 1
            shutil.rmtree(class_dir, ignore_errors=True)
    return moved


def remove_disabled_detector_classes_from_working_copy(dataset):
    dataset = Path(dataset)
    detector_root = dataset / "detector_dataset"
    data_yaml = detector_root / "data.yaml"
    if not data_yaml.exists():
        return {"removed_labels": 0, "emptied_label_files": 0, "names": []}
    data = yaml.safe_load(data_yaml.read_text())
    old_names = load_yaml_names(data)
    kept_names = [name for name in old_names if name not in DISABLED_DETECTOR_CLASSES]
    remap = {old_id: kept_names.index(name) for old_id, name in enumerate(old_names) if name in kept_names}
    removed = 0
    emptied = 0
    for split in ("train", "val", "test"):
        label_dir = detector_root / "labels" / split
        if not label_dir.exists():
            continue
        for label_path in sorted(label_dir.glob("*.txt")):
            original = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
            kept = []
            for line in original:
                parts = line.split()
                try:
                    old_id = int(float(parts[0]))
                except Exception:
                    removed += 1
                    continue
                old_name = old_names[old_id] if 0 <= old_id < len(old_names) else None
                if old_name in DISABLED_DETECTOR_CLASSES or old_id not in remap:
                    removed += 1
                    continue
                kept.append(" ".join([str(remap[old_id]), *parts[1:]]))
            if kept:
                label_path.write_text("\n".join(kept) + "\n")
            else:
                if original:
                    emptied += 1
                label_path.write_text("")
    data["names"] = {i: name for i, name in enumerate(kept_names)}
    data["nc"] = len(kept_names)
    data_yaml.write_text(yaml.safe_dump(data, sort_keys=False))
    return {"removed_labels": removed, "emptied_label_files": emptied, "names": kept_names}


def update_runtime_classifier_manifests(dataset):
    dataset = Path(dataset)
    train_root = dataset / "classifier_dataset" / "train"
    if not train_root.exists():
        return []
    active_dirs = sorted(p.name for p in train_root.iterdir() if p.is_dir() and any(x.suffix.lower() in IMAGE_EXTS for x in p.iterdir()))
    manifests = dataset / "manifests"
    manifests.mkdir(parents=True, exist_ok=True)
    (manifests / "classifier_classes_runtime.json").write_text(json.dumps(active_dirs, indent=2))
    display = {}
    for name in active_dirs:
        display[name] = name.replace("_", " ") if name != "not_target" else "not_target"
    (manifests / "classifier_display_names_runtime.json").write_text(json.dumps(display, indent=2))
    return active_dirs


def apply_runtime_dataset_policy(dataset):
    classifier_moved = move_disabled_classifier_classes_to_not_target(dataset)
    detector_summary = remove_disabled_detector_classes_from_working_copy(dataset)
    active_classifier_dirs = update_runtime_classifier_manifests(dataset)
    summary = {
        "disabled_classifier_moved_to_not_target": classifier_moved,
        "disabled_detector_cleanup": detector_summary,
        "active_classifier_dir_count": len(active_classifier_dirs),
    }
    summary_path = Path(dataset) / "manifests" / "runtime_training_policy_summary.json"
    summary_path.write_text(json.dumps(summary, indent=2))
    print("Runtime training dataset policy applied:")
    print(json.dumps(summary, indent=2)[:4000])
    return summary

def find_two_stage_dataset():
    candidates = [
        Path("/kaggle/input/datasets/ashok205/hearsight-ts-dataset-v3/hearsight_two_stage_dataset"),
        Path("/kaggle/input/hearsight-ts-dataset-v3/hearsight_two_stage_dataset"),
        Path("/kaggle/input/hearsight_ts_dataset_v3/hearsight_two_stage_dataset"),
        Path("/kaggle/input/datasets/ashok205/hearsight-two-stage-dataset-v2/hearsight_two_stage_dataset"),
        Path("/kaggle/input/hearsight-two-stage-dataset-v2/hearsight_two_stage_dataset"),
        Path("/kaggle/input/hearsight_two_stage_dataset_v2/hearsight_two_stage_dataset"),
        Path("/kaggle/working/hearsight_two_stage_dataset"),
    ]
    if Path("/kaggle/input").exists():
        candidates.extend(p for p in Path("/kaggle/input").glob("**/hearsight_two_stage_dataset") if p.is_dir())
        candidates.extend(p.parent.parent for p in Path("/kaggle/input").glob("**/detector_dataset/data.yaml"))
    for p in candidates:
        if (p / "detector_dataset" / "data.yaml").exists() and (p / "classifier_dataset").exists():
            return p
    raise FileNotFoundError("Attach the two-stage dataset created by the dataset builder notebook.")

def prepare_writable_dataset(source):
    source = Path(source)
    if not Path("/kaggle").exists():
        return source
    target = Path("/kaggle/working/hearsight_two_stage_dataset_train")
    marker = target / ".source_path"
    if target.exists() and marker.exists() and marker.read_text() == str(source):
        print("Using existing writable dataset copy:", target)
    else:
        if target.exists():
            shutil.rmtree(target)
        print("Copying dataset to writable working directory with cp -a...")
        t0 = time.time()
        subprocess.run(["cp", "-a", str(source), str(target)], check=True)
        marker.write_text(str(source))
        print(f"Dataset copy complete in {time.time() - t0:.1f}s -> {target}")
    for cache_file in target.rglob("*.cache"):
        try:
            cache_file.unlink()
        except FileNotFoundError:
            pass
    removed_tiny = 0
    try:
        from PIL import Image
        image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        for img_path in (target / "classifier_dataset").rglob("*"):
            if img_path.suffix.lower() not in image_exts:
                continue
            try:
                with Image.open(img_path) as im:
                    w, h = im.size
                if min(w, h) < 10:
                    img_path.unlink()
                    removed_tiny += 1
            except Exception:
                img_path.unlink(missing_ok=True)
                removed_tiny += 1
    except Exception as e:
        print("Tiny-image cleanup skipped:", e)
    print("Removed tiny/corrupt classifier images from working copy:", removed_tiny)
    apply_runtime_dataset_policy(target)
    det_yaml = target / "detector_dataset" / "data.yaml"
    data = yaml.safe_load(det_yaml.read_text())
    data["path"] = "."
    data["train"] = "images/train"
    data["val"] = "images/val"
    data["test"] = "images/test"
    det_yaml.write_text(yaml.safe_dump(data, sort_keys=False))
    return target

SOURCE_DATASET = find_two_stage_dataset()
DATASET = prepare_writable_dataset(SOURCE_DATASET)
PROJECT = Path("/kaggle/working/runs/hearsight_two_stage") if Path("/kaggle").exists() else Path("runs/hearsight_two_stage")
print("SOURCE_DATASET =", SOURCE_DATASET)
print("TRAINING_DATASET =", DATASET)
print("PROJECT =", PROJECT)


Copying dataset to writable working directory with cp -a...
Dataset copy complete in 493.7s -> /kaggle/working/hearsight_two_stage_dataset_train
Removed tiny/corrupt classifier images from working copy: 0
Runtime training dataset policy applied:
{
  "disabled_classifier_moved_to_not_target": {
    "train": {
      "Exit": 323,
      "Fire extinguisher": 837,
      "Traffic light red": 350,
      "Traffic light yellow": 14,
      "Traffic light green": 332
    },
    "val": {
      "Exit": 170,
      "Fire extinguisher": 1219,
      "Traffic light red": 44,
      "Traffic light yellow": 3,
      "Traffic light green": 35
    },
    "test": {
      "Exit": 143,
      "Fire extinguisher": 784,
      "Traffic light red": 45,
      "Traffic light yellow": 2,
      "Traffic light green": 48
    }
  },
  "disabled_detector_cleanup": {
    "removed_labels": 3244,
    "emptied_label_files": 766,
    "names": [
      "road_sign",
      "facility_sign",
      "medical_sign"
    ]
  },
  "active_c

In [5]:

from pathlib import Path

def parse_batch(value):
    if str(value) == "auto":
        return "auto"
    try:
        return int(value)
    except ValueError:
        return float(value)

def make_runtime_detector_yaml(dataset):
    import yaml
    dataset = Path(dataset)
    detector_root = dataset / "detector_dataset"
    if not (detector_root / "images" / "val").exists():
        raise FileNotFoundError(f"Detector val images folder missing: {detector_root / 'images' / 'val'}")
    source_yaml = detector_root / "data.yaml"
    data = yaml.safe_load(source_yaml.read_text())
    data["path"] = str(detector_root)
    data["train"] = "images/train"
    data["val"] = "images/val"
    data["test"] = "images/test"
    runtime_dir = Path("/kaggle/working/runtime_yamls") if Path("/kaggle").exists() else Path("runtime_yamls")
    runtime_dir.mkdir(parents=True, exist_ok=True)
    runtime_yaml = runtime_dir / "detector_data.yaml"
    runtime_yaml.write_text(yaml.safe_dump(data, sort_keys=False))
    print("Runtime detector yaml:", runtime_yaml)
    print(runtime_yaml.read_text())
    return runtime_yaml


def train_detector(dataset, project, model_path, epochs, imgsz, batch, device, workers, cache_mode, final_test):
    from ultralytics import YOLO
    detector_yaml = make_runtime_detector_yaml(dataset)
    model = YOLO(model_path)
    model.train(
        data=str(detector_yaml), epochs=epochs, imgsz=imgsz, batch=parse_batch(batch), device=device,
        project=str(project), name="generic_detector", pretrained=True, optimizer="AdamW",
        lr0=0.002, lrf=0.01, warmup_epochs=5, close_mosaic=20, mosaic=0.025,
        mixup=0.0, copy_paste=0.0, auto_augment=None, erasing=0.0,
        patience=30, cache=cache_mode, workers=workers, plots=False, amp=True, cos_lr=True,
    )
    if final_test:
        model.val(data=str(detector_yaml), split="test", imgsz=imgsz, conf=0.45, iou=0.55,
                  project=str(project), name="generic_detector_test")

def train_classifier(dataset, project, model_path, epochs, imgsz, batch, device, workers, cache_mode, final_test):
    from ultralytics import YOLO
    classifier_dir = Path(dataset) / "classifier_dataset"
    model = YOLO(model_path)
    model.train(
        data=str(classifier_dir), epochs=epochs, imgsz=imgsz, batch=parse_batch(batch), device=device,
        project=str(project), name="crop_classifier", pretrained=True, optimizer="AdamW",
        lr0=globals().get("CLASSIFIER_LR0", 0.0015), lrf=globals().get("CLASSIFIER_LRF", 0.01), patience=20, cache=cache_mode, workers=workers, plots=False, amp=True, cos_lr=True,
    )
    if final_test:
        model.val(data=str(classifier_dir), split="test", imgsz=imgsz,
                  project=str(project), name="crop_classifier_test")


In [6]:
# Training mode:
#   "detector" = train only the generic detector, recommended after rebuilding detector_dataset_v3
#   "classifier" = train only the crop classifier
#   "both" = train detector and classifier in parallel
TRAIN_MODE = "classifier"

# Keep nano models for Raspberry Pi deployment.
DETECTOR_MODEL = "yolo26n.pt"
CLASSIFIER_MODEL = "yolo26n-cls.pt"

# Detector-only retrain strategy for small/far signs.
DETECTOR_EPOCHS = 100
CLASSIFIER_EPOCHS = 100

DETECTOR_IMGSZ = 640
CLASSIFIER_IMGSZ = 640

# Detector-only can use both T4s with a larger global batch.
# If DDP causes a Kaggle issue, set DETECTOR_DEVICE = 0 and DETECTOR_BATCH = 16.
DETECTOR_BATCH = 196
CLASSIFIER_BATCH = 196
DETECTOR_DEVICE = [0,1]
CLASSIFIER_DEVICE = [0,1]
CLASSIFIER_WORKERS = 1

# Classifier defaults for two T4 GPUs at lower crop resolution.
CLASSIFIER_LR0 = 0.001
CLASSIFIER_LRF = 0.01

WORKERS = CLASSIFIER_WORKERS if TRAIN_MODE=='classifier' else 2
DETECTOR_CACHE_MODE = False
CLASSIFIER_CACHE_MODE = False
PREBUILD_CACHES = False
FINAL_TEST = True


In [7]:
import os, subprocess, sys, textwrap, threading, time
from pathlib import Path

def print_gpu_status():
    try:
        out = subprocess.check_output([
            "nvidia-smi",
            "--query-gpu=index,memory.used,memory.total,utilization.gpu",
            "--format=csv,noheader,nounits",
        ], text=True).strip()
        print("gpu_status[index, used_MB, total_MB, util_%]:")
        print(out)
    except Exception as e:
        print("gpu_status unavailable:", e)


def write_runtime_train_script(stage):
    script_dir = Path('/kaggle/working/train_scripts') if Path('/kaggle').exists() else Path('train_scripts')
    script_dir.mkdir(parents=True, exist_ok=True)
    if stage == 'detector':
        script = f'''
import os
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
from pathlib import Path
import yaml
from ultralytics import YOLO
from torch.utils.data import DataLoader as _TorchDataLoader
import ultralytics.data.build as _ultra_build

class _LeanDataLoader(_TorchDataLoader):
    def __init__(self, *args, **kwargs):
        kwargs['pin_memory'] = False
        kwargs['persistent_workers'] = False
        super().__init__(*args, **kwargs)

_ultra_build.DataLoader = _LeanDataLoader

dataset = Path(r"{str(DATASET)}")
project = Path(r"{str(PROJECT)}")
detector_root = dataset / 'detector_dataset'
data = yaml.safe_load((detector_root / 'data.yaml').read_text())
data['path'] = str(detector_root)
data['train'] = 'images/train'
data['val'] = 'images/val'
data['test'] = 'images/test'
runtime_dir = Path('/kaggle/working/runtime_yamls') if Path('/kaggle').exists() else Path('runtime_yamls')
runtime_dir.mkdir(parents=True, exist_ok=True)
runtime_yaml = runtime_dir / 'detector_data.yaml'
runtime_yaml.write_text(yaml.safe_dump(data, sort_keys=False))
print('[DETECTOR] Runtime YAML:', runtime_yaml, flush=True)
print(runtime_yaml.read_text(), flush=True)

model = YOLO(r"{str(Path(DETECTOR_MODEL).resolve())}")
model.train(data=str(runtime_yaml), epochs={DETECTOR_EPOCHS}, imgsz={DETECTOR_IMGSZ}, batch={DETECTOR_BATCH}, device={DETECTOR_DEVICE}, project=str(project), name='generic_detector', exist_ok=True, pretrained=True, optimizer='AdamW', lr0=0.0005, lrf=0.01, warmup_epochs=4, close_mosaic=10, mosaic=0.15, mixup=0.0, copy_paste=0.0, auto_augment=None, erasing=0.0, patience=20, cache=False, workers={WORKERS}, plots=False, amp=True, cos_lr=True)
if {FINAL_TEST!r}:
    model.val(data=str(runtime_yaml), split='test', imgsz={DETECTOR_IMGSZ}, conf=0.45, iou=0.55, project=str(project), name='generic_detector_test')
print('[DETECTOR] complete', flush=True)
'''
    elif stage == 'classifier':
        script = f'''
import os
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
from pathlib import Path
from ultralytics import YOLO
from torch.utils.data import DataLoader as _TorchDataLoader
import ultralytics.data.build as _ultra_build

class _LeanDataLoader(_TorchDataLoader):
    def __init__(self, *args, **kwargs):
        kwargs['pin_memory'] = False
        kwargs['persistent_workers'] = False
        super().__init__(*args, **kwargs)

_ultra_build.DataLoader = _LeanDataLoader

dataset = Path(r"{str(DATASET)}")
project = Path(r"{str(PROJECT)}")
classifier_dir = dataset / 'classifier_dataset'
model = YOLO(r"{str(Path(CLASSIFIER_MODEL).resolve())}")
model.train(data=str(classifier_dir), epochs={CLASSIFIER_EPOCHS}, imgsz={CLASSIFIER_IMGSZ}, batch={CLASSIFIER_BATCH}, device={CLASSIFIER_DEVICE!r}, project=str(project), name='crop_classifier', exist_ok=True, pretrained=True, optimizer='AdamW', lr0={CLASSIFIER_LR0}, lrf={CLASSIFIER_LRF}, patience=20, cache=False, workers={WORKERS}, plots=False, amp=True, cos_lr=True)
if {FINAL_TEST!r}:
    model.val(data=str(classifier_dir), split='test', imgsz={CLASSIFIER_IMGSZ}, project=str(project), name='crop_classifier_test')
print('[CLASSIFIER] complete', flush=True)
'''
    else:
        raise ValueError(stage)
    path = script_dir / f'train_{stage}.py'
    path.write_text(textwrap.dedent(script))
    return path

def stream_log(name, log_path, stop_event):
    last_pos = 0
    prefix = f'[{name.upper()}] '
    keep_tokens = ('Epoch', 'GPU_mem', 'Starting training', 'complete', 'Results saved', 'error', 'Error', 'Traceback', 'RuntimeError', 'WARNING', 'mAP', 'accuracy', 'top1', 'top5')
    while not stop_event.is_set():
        if log_path.exists():
            with log_path.open('r', errors='ignore') as f:
                f.seek(last_pos)
                for line in f:
                    text = line.rstrip()
                    if any(tok in text for tok in keep_tokens):
                        print(prefix + text)
                last_pos = f.tell()
        time.sleep(5)

def write_metadata_cache_script(stage):
    script_dir = Path('/kaggle/working/train_scripts') if Path('/kaggle').exists() else Path('train_scripts')
    script_dir.mkdir(parents=True, exist_ok=True)
    if stage == 'detector':
        script = f'''
import os
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
from pathlib import Path
import yaml
from ultralytics.cfg import DEFAULT_CFG, get_cfg
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset

dataset = Path(r"{str(DATASET)}")
detector_root = dataset / 'detector_dataset'
data = yaml.safe_load((detector_root / 'data.yaml').read_text())
data['path'] = str(detector_root)
data['train'] = 'images/train'
data['val'] = 'images/val'
data['test'] = 'images/test'
runtime_dir = Path('/kaggle/working/runtime_yamls') if Path('/kaggle').exists() else Path('runtime_yamls')
runtime_dir.mkdir(parents=True, exist_ok=True)
runtime_yaml = runtime_dir / 'detector_data.yaml'
runtime_yaml.write_text(yaml.safe_dump(data, sort_keys=False))
checked = check_det_dataset(str(runtime_yaml))
args = get_cfg(DEFAULT_CFG)
args.imgsz = {DETECTOR_IMGSZ}
args.cache = False
args.rect = False
args.single_cls = False
args.classes = None
args.fraction = 1.0
for split in ('train', 'val', 'test'):
    print(f'[CACHE][DETECTOR] building {{split}} metadata cache', flush=True)
    YOLODataset(img_path=checked[split], imgsz=args.imgsz, batch_size={DETECTOR_BATCH}, augment=False, hyp=args, rect=False, cache=False, single_cls=False, data=checked, task='detect', prefix=f'precache {{split}}: ')
print('[CACHE][DETECTOR] complete', flush=True)
'''
    elif stage == 'classifier':
        script = f'''
import os
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
from pathlib import Path
from ultralytics.cfg import DEFAULT_CFG, get_cfg
from ultralytics.data.dataset import ClassificationDataset

classifier_root = Path(r"{str(DATASET)}") / 'classifier_dataset'
args = get_cfg(DEFAULT_CFG)
args.imgsz = {CLASSIFIER_IMGSZ}
args.cache = False
args.fraction = 1.0
for split in ('train', 'val', 'test'):
    print(f'[CACHE][CLASSIFIER] building {{split}} metadata cache', flush=True)
    ClassificationDataset(root=str(classifier_root / split), args=args, augment=False, prefix=f'precache {{split}}')
print('[CACHE][CLASSIFIER] complete', flush=True)
'''
    else:
        raise ValueError(stage)
    path = script_dir / f'cache_{stage}.py'
    path.write_text(textwrap.dedent(script))
    return path

def prebuild_metadata_caches():
    logs_dir = Path('/kaggle/working/logs') if Path('/kaggle').exists() else Path('logs')
    logs_dir.mkdir(parents=True, exist_ok=True)
    for stage in stages:
        log_path = logs_dir / f'cache_{stage}.log'
        print(f'Prebuilding {stage} metadata cache sequentially. Full log: {log_path}')
        with log_path.open('w', buffering=1) as log_f:
            proc = subprocess.run([sys.executable, str(write_metadata_cache_script(stage))], stdout=log_f, stderr=subprocess.STDOUT)
        if proc.returncode != 0:
            tail = log_path.read_text(errors='ignore').splitlines()[-80:]
            print('\n'.join(tail))
            raise RuntimeError(f'{stage} metadata cache prebuild failed with exit code {proc.returncode}')
        lines = log_path.read_text(errors='ignore').splitlines()
        summary = [line for line in lines if '[CACHE]' in line or 'WARNING' in line or 'ERROR' in line or 'Traceback' in line]
        for line in summary[-12:]:
            print(line)

def launch(stage):
    script = write_runtime_train_script(stage)
    logs_dir = Path('/kaggle/working/logs') if Path('/kaggle').exists() else Path('logs')
    logs_dir.mkdir(parents=True, exist_ok=True)
    log_path = logs_dir / f'train_{stage}.log'
    log_f = open(log_path, 'w', buffering=1)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    proc = subprocess.Popen([sys.executable, str(script)], stdout=log_f, stderr=subprocess.STDOUT, bufsize=1, env=env)
    stop = threading.Event()
    thread = threading.Thread(target=stream_log, args=(stage, log_path, stop), daemon=True)
    thread.start()
    return proc, stop, log_f

if TRAIN_MODE == 'detector':
    stages = ('detector',)
elif TRAIN_MODE == 'classifier':
    stages = ('classifier',)
elif TRAIN_MODE == 'both':
    stages = ('detector', 'classifier')
else:
    raise ValueError(f'Unknown TRAIN_MODE: {TRAIN_MODE}')

if PREBUILD_CACHES:
    prebuild_metadata_caches()
else:
    print('Skipping metadata cache prebuild.')

print('TRAIN_MODE =', TRAIN_MODE)
print('stages =', stages)
print_gpu_status()
jobs = {stage: launch(stage) for stage in stages}
last_status = 0
while True:
    exit_codes = {name: proc.poll() for name, (proc, _, _) in jobs.items()}
    if all(code is not None for code in exit_codes.values()):
        break
    failed = {name: code for name, code in exit_codes.items() if code not in (None, 0)}
    if failed:
        print('failure detected:', failed)
        for name, (proc, _, _) in jobs.items():
            if proc.poll() is None:
                proc.terminate()
        break
    now = time.time()
    if now - last_status >= 300:
        print('running...', exit_codes)
        print_gpu_status()
        last_status = now
    time.sleep(60)

final_codes = {}
for name, (proc, stop, log_f) in jobs.items():
    proc.wait()
    final_codes[name] = proc.returncode
    stop.set()
    log_f.close()
print('final exit codes:', final_codes)
if any(code != 0 for code in final_codes.values()):
    raise RuntimeError(f'Training failed: {final_codes}')
print('Training completed.')


Skipping metadata cache prebuild.
TRAIN_MODE = classifier
stages = ('classifier',)
gpu_status[index, used_MB, total_MB, util_%]:
0, 0, 15360, 0
1, 0, 15360, 0
running... {'classifier': None}
gpu_status[index, used_MB, total_MB, util_%]:
0, 0, 15360, 0
1, 0, 15360, 0
[CLASSIFIER] Starting training for 100 epochs...
[CLASSIFIER]       Epoch    GPU_mem       loss  Instances       Size
running... {'classifier': None}
gpu_status[index, used_MB, total_MB, util_%]:
0, 9259, 15360, 0
1, 9247, 15360, 56
[CLASSIFIER]                classes   top1_acc   top5_acc: 8% ━─────────── 1/12 6.3s/it 1.9s<1:09
[CLASSIFIER]                classes   top1_acc   top5_acc: 17% ━━────────── 2/12 1.0s/it 2.2s<10.4s
[CLASSIFIER]                classes   top1_acc   top5_acc: 25% ━━━───────── 3/12 1.5it/s 2.6s<6.0s
[CLASSIFIER]                classes   top1_acc   top5_acc: 33% ━━━━──────── 4/12 1.9it/s 2.9s<4.2s
[CLASSIFIER]                classes   top1_acc   top5_acc: 42% ━━━━━─────── 5/12 1.4it/s 7.7s<5.0s
[CLAS

In [8]:
from pathlib import Path
for p in PROJECT.glob("**/weights/best.pt"):
    print(p)


/kaggle/working/runs/hearsight_two_stage/crop_classifier/weights/best.pt
